# Imports

In [1]:
import os
import cv2
import random
import shutil
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers


# Data Path

In [6]:
RAW_DIR   = "/content/drive/MyDrive/CVPR_practice/data"
FACE_DIR  = "/content/drive/MyDrive/CVPR_practice/Dataset_Faces"


# Image Size

In [8]:
IMG_SIZE = (224, 224)
MIN_FACE_SIZE = (80, 80)

# For Face detect from image

In [9]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Extract from path folder

In [11]:
os.makedirs(FACE_DIR, exist_ok=True)

print("Face extraction")

class_count = 0                 # counts valid classes
valid_students = []             # optional: keep mapping

for student in sorted(os.listdir(RAW_DIR)):
    student_path = os.path.join(RAW_DIR, student)
    if not os.path.isdir(student_path):
        continue

    # collect valid image files first
    img_files = [f for f in os.listdir(student_path)
                 if f.lower().endswith((".jpg", ".png", ".jpeg"))]

    # IGNORE folder if it has zero images
    if len(img_files) == 0:
        print(f"IGNORED (no images): {student}")
        continue

    out_dir = os.path.join(FACE_DIR, student)
    os.makedirs(out_dir, exist_ok=True)

    face_count = 0

    for img_name in img_files:
        img_path = os.path.join(student_path, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray, scaleFactor=1.2, minNeighbors=5, minSize=MIN_FACE_SIZE
        )

        if len(faces) == 0:
            continue

        # take largest face
        faces = sorted(faces, key=lambda x: x[2] * x[3], reverse=True)
        x, y, w, h = faces[0]

        face = img[y:y+h, x:x+w]
        face = cv2.resize(face, IMG_SIZE)

        cv2.imwrite(
            os.path.join(out_dir, f"{student}_{face_count}.jpg"),
            face
        )
        face_count += 1

    # IGNORE folder if no faces were detected
    if face_count == 0:
        print(f"IGNORED (no faces detected): {student}")
        os.rmdir(out_dir)  # remove empty output folder
        continue

    class_count += 1
    valid_students.append(student)

    print(f"CLASS {class_count:02d} | {student}: {face_count} faces saved")

print(f"\nFace extraction complete")
print(f"Total valid classes: {class_count}")


Face extraction
CLASS 01 | 21-45902-3: 6 faces saved
CLASS 02 | 22-46138-1: 17 faces saved
CLASS 03 | 22-46139-1: 14 faces saved
CLASS 04 | 22-46141-1: 20 faces saved
CLASS 05 | 22-46258-1: 8 faces saved
CLASS 06 | 22-46275-1: 20 faces saved
CLASS 07 | 22-46293-1: 16 faces saved
CLASS 08 | 22-46342-1: 18 faces saved
CLASS 09 | 22-46473-1: 18 faces saved
CLASS 10 | 22-46536-1: 17 faces saved
CLASS 11 | 22-46590-1: 15 faces saved
CLASS 12 | 22-46666-1: 20 faces saved
CLASS 13 | 22-46679-1: 15 faces saved
CLASS 14 | 22-46887-1: 15 faces saved
CLASS 15 | 22-46931-1: 10 faces saved
CLASS 16 | 22-46983-1: 14 faces saved
CLASS 17 | 22-47180-1: 19 faces saved
IGNORED (no images): 22-47542-2
CLASS 18 | 22-47813-2: 10 faces saved
IGNORED (no images): 22-47884-2
CLASS 19 | 22-47888-2: 18 faces saved
CLASS 20 | 22-47892-2: 8 faces saved
CLASS 21 | 22-47898-2: 16 faces saved
CLASS 22 | 22-47968-2: 16 faces saved
IGNORED (no images): 22-48005-2
CLASS 23 | 22-48021-2: 12 faces saved
CLASS 24 | 22-480

# TRAIN / TEST SPLIT directions

In [13]:
INPUT_DIR  = "/content/drive/MyDrive/CVPR_practice/Dataset_Faces"
OUTPUT_DIR = "/content/drive/MyDrive/CVPR_practice/Final_Dataset"

NUM_CLASSES = class_count
IMG_SIZE = (224,224)
TARGET_IMAGES_PER_CLASS = 100
TRAIN_RATIO = 0.8

random.seed(42)
np.random.seed(42)


In [19]:
print(len(valid_students))

62


# TRAIN / TEST SPLIT

In [23]:
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

print("Train-Test split started\n")


for label, student in enumerate(valid_students):
    student_path = os.path.join(FACE_DIR, student)

    images = [f for f in os.listdir(student_path)
              if f.lower().endswith((".jpg",".png",".jpeg"))]

    random.shuffle(images)
    split = int(len(images) * TRAIN_RATIO)

    train_imgs = images[:split]
    test_imgs  = images[split:]

    for split_name, img_list in zip(["train_raw", "test"],
                                    [train_imgs, test_imgs]):
        out_dir = os.path.join(OUTPUT_DIR, split_name, str(label))
        ensure_dir(out_dir)

        for img in img_list:
            shutil.copy(
                os.path.join(student_path, img),
                os.path.join(out_dir, img)
            )

print("Train-Test split complete\n")


Train-Test split started...

Train-Test split complete



# OFFLINE AUGMENTATION (BALANCING)

In [24]:
def augment(img):
    aug = []

    aug.append(cv2.flip(img, 1))

    for angle in [-10, 10]:
        M = cv2.getRotationMatrix2D((112,112), angle, 1)
        aug.append(cv2.warpAffine(img, M, IMG_SIZE))

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hsv[:,:,2] = np.clip(hsv[:,:,2] * 1.2, 0, 255)
    aug.append(cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR))

    return aug


In [25]:
print("Offline augmentation started\n")

RAW_TRAIN   = os.path.join(OUTPUT_DIR, "train_raw")
FINAL_TRAIN = os.path.join(OUTPUT_DIR, "train")

for cls in range(NUM_CLASSES):
    in_dir  = os.path.join(RAW_TRAIN, str(cls))
    out_dir = os.path.join(FINAL_TRAIN, str(cls))
    ensure_dir(out_dir)

    imgs = [f for f in os.listdir(in_dir)
            if f.lower().endswith((".jpg",".png",".jpeg"))]

    if len(imgs) == 0:
        print(f"Skipping class {cls} (no training images)")
        continue

    saved = 0

    for img_name in imgs:
        img = cv2.imread(os.path.join(in_dir, img_name))
        img = cv2.resize(img, IMG_SIZE)
        cv2.imwrite(os.path.join(out_dir, img_name), img)
        saved += 1

    idx = 0
    while saved < TARGET_IMAGES_PER_CLASS:
        img = cv2.imread(os.path.join(in_dir, imgs[idx % len(imgs)]))
        img = cv2.resize(img, IMG_SIZE)

        for a in augment(img):
            if saved >= TARGET_IMAGES_PER_CLASS:
                break
            cv2.imwrite(os.path.join(out_dir, f"aug_{saved}.jpg"), a)
            saved += 1

        idx += 1

    print(f"Class {cls}: {saved} images")

print("\nAugmentation complete ")


Offline augmentation started...

Class 0: 100 images
Class 1: 100 images
Class 2: 100 images
Class 3: 100 images
Class 4: 100 images
Class 5: 100 images
Class 6: 100 images
Class 7: 100 images
Class 8: 100 images
Class 9: 100 images
Class 10: 100 images
Class 11: 100 images
Class 12: 100 images
Class 13: 100 images
Class 14: 100 images
Class 15: 100 images
Class 16: 100 images
Class 17: 100 images
Class 18: 100 images
Class 19: 100 images
Class 20: 100 images
Class 21: 100 images
Class 22: 100 images
Class 23: 100 images
Class 24: 100 images
Class 25: 100 images
Class 26: 100 images
Class 27: 100 images
Class 28: 100 images
Class 29: 100 images
Class 30: 100 images
Class 31: 100 images
Class 32: 100 images
Class 33: 100 images
Class 34: 100 images
Class 35: 100 images
Class 36: 100 images
Class 37: 100 images
Class 38: 100 images
Class 39: 100 images
Class 40: 100 images
Class 41: 100 images
Class 42: 100 images
Class 43: 100 images
Class 44: 100 images
Class 45: 100 images
Class 46: 1

# LOAD DATASET

In [26]:
def load_dataset(path):
    X, Y = [], []

    for cls in range(NUM_CLASSES):
        cls_dir = os.path.join(path, str(cls))
        if not os.path.exists(cls_dir):
            continue

        for img in os.listdir(cls_dir):
            img_path = os.path.join(cls_dir, img)
            im = cv2.imread(img_path)
            if im is None:
                continue

            im = cv2.resize(im, IMG_SIZE)
            im = im / 255.0

            X.append(im)
            Y.append(cls)

    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.int32)

    idx = np.random.permutation(len(X))
    return X[idx], Y[idx]

X_train, Y_train = load_dataset(os.path.join(OUTPUT_DIR,"train"))
X_test,  Y_test  = load_dataset(os.path.join(OUTPUT_DIR,"test"))

print("Train:", X_train.shape, Y_train.shape)
print("Test :", X_test.shape, Y_test.shape)


Train: (6200, 224, 224, 3) (6200,)
Test : (221, 224, 224, 3) (221,)


# Model

In [27]:
model = keras.Sequential([
    keras.Input(shape=(224,224,3)),

    layers.Conv2D(32,3,padding="same",activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64,3,padding="same",activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128,3,padding="same",activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(256,3,padding="same",activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256,activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(NUM_CLASSES,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    12,845,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 62)             │        15,934 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,249,662 (50.54 MB)

 Trainable params: 13,249,662 (50.54 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train, Y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    shuffle=True
)


Epoch 1/25
155/155 ━━━━━━━━━━━━━━━━━━━━ 24s 98ms/step - accuracy: 0.0273 - loss: 4.1351 - val_accuracy: 0.4024 - val_loss: 2.6347
Epoch 2/25
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 69ms/step - accuracy: 0.4205 - loss: 2.2814 - val_accuracy: 0.8460 - val_loss: 0.6924
Epoch 3/25
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.7300 - loss: 0.9718 - val_accuracy: 0.9218 - val_loss: 0.3516
Epoch 4/25
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.8469 - loss: 0.5432 - val_accuracy: 0.9403 - val_loss: 0.2426
Epoch 5/25
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.8853 - loss: 0.3830 - val_accuracy: 0.9589 - val_loss: 0.1645
Epoch 6/25
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 69ms/step - accuracy: 0.9232 - loss: 0.2596 - val_accuracy: 0.9669 - val_loss: 0.1374
Epoch 7/25
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.9349 - loss: 0.2115 - val_accuracy: 0.9694 - val_loss: 0.1203
Epoch 8/25
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.9460 - loss: 0.1814 - 

# Save Model

In [32]:

MODEL_PATH = "/content/drive/MyDrive/CVPR_practice/Attendance_Model.keras"
model.save(MODEL_PATH)
print("Model saved ")


Model saved 


# save the labels

In [31]:
import os
import json

FACE_DIR = "/content/drive/MyDrive/CVPR_practice/Dataset_Faces"
label_map = {}

label = 0
for student in sorted(os.listdir(FACE_DIR)):
    student_path = os.path.join(FACE_DIR, student)

    images = [f for f in os.listdir(student_path)
              if f.lower().endswith((".jpg",".png",".jpeg"))]

    if len(images) == 0:
        continue

    label_map[label] = student
    label += 1

print("Recovered label_map:")
print(label_map)

# Save it
with open("/content/drive/MyDrive/CVPR_practice/label_map.json", "w") as f:
    json.dump(label_map, f, indent=4)


Recovered label_map:
{0: '21-45902-3', 1: '22-46138-1', 2: '22-46139-1', 3: '22-46141-1', 4: '22-46258-1', 5: '22-46275-1', 6: '22-46293-1', 7: '22-46342-1', 8: '22-46473-1', 9: '22-46536-1', 10: '22-46590-1', 11: '22-46666-1', 12: '22-46679-1', 13: '22-46887-1', 14: '22-46931-1', 15: '22-46983-1', 16: '22-47180-1', 17: '22-47813-2', 18: '22-47888-2', 19: '22-47892-2', 20: '22-47898-2', 21: '22-47968-2', 22: '22-48021-2', 23: '22-48023-2', 24: '22-48055-2', 25: '22-48064-2', 26: '22-48091-2', 27: '22-48133-2', 28: '22-48205-2', 29: '22-48434-3', 30: '22-48569-3', 31: '22-48582-3', 32: '22-48833-3', 33: '22-49037-3', 34: '22-49167-3', 35: '22-49196-3', 36: '22-49338-3', 37: '22-49355-3', 38: '22-49370-3', 39: '22-49421-3', 40: '22-49450-3', 41: '22-49451-3', 42: '22-49453-3', 43: '22-49507-3', 44: '22-49538-3', 45: '22-49575-3', 46: '22-49609-3', 47: '22-49745-3', 48: '22-49783-3', 49: '22-49791-3', 50: '22-49800-3', 51: '22-49824-3', 52: '22-49843-3', 53: '22-49852-3', 54: '22-49862-3'